# CNN training handoff (baseline and GA-CLAHE refinement)
This notebook imports the exact uncommitted `cnn/` implementation from the separately uploaded `clahe-cnn-source.zip`; it does not clone GitHub or duplicate the architecture. Attach that ZIP and the LOL dataset as two separate Kaggle inputs, select a GPU accelerator, and edit only the two explicit input paths below. The packaged split manifest is copied unchanged into `/kaggle/working`. `eval15` remains final-test-only.

In [ ]:
from pathlib import Path
import shutil, sys, zipfile, torch

SOURCE_ZIP = Path('/kaggle/input/clahe-cnn-source/clahe-cnn-source.zip')  # uploaded source ZIP
DATASET_ROOT = Path('/kaggle/input/lol-dataset/Dataset')                  # contains our485/ and eval15/
WORK_ROOT = Path('/kaggle/working/clahe-cnn')
PROJECT_ROOT = WORK_ROOT / 'source'
MANIFEST = WORK_ROOT / 'splits/our485_split.json'
RESUME_BASELINE = None      # e.g. Path('/kaggle/input/my-checkpoints/baseline-latest.pt')
RESUME_REFINEMENT = None    # e.g. Path('/kaggle/input/my-checkpoints/refinement-latest.pt')
RUN_BASELINE = False        # opt in after the checks below
RUN_REFINEMENT = False      # expensive: includes reference-free GA preprocessing
RUN_FINAL_TEST = False      # enable only after model selection is finished

assert SOURCE_ZIP.is_file(), f'Attach the source ZIP and update SOURCE_ZIP: {SOURCE_ZIP}'
assert (DATASET_ROOT / 'our485/low').is_dir(), DATASET_ROOT
WORK_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    unsafe = [name for name in archive.namelist() if Path(name).is_absolute() or '..' in Path(name).parts]
    assert not unsafe, f'Unsafe ZIP entries: {unsafe}'
    archive.extractall(PROJECT_ROOT)
assert (PROJECT_ROOT / 'cnn/model.py').is_file(), PROJECT_ROOT
packaged_manifest = PROJECT_ROOT / 'splits/our485_split.json'
assert packaged_manifest.is_file(), packaged_manifest
MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(packaged_manifest, MANIFEST)  # exact saved split; do not regenerate or edit
sys.path.insert(0, str(PROJECT_ROOT))
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('Source:', PROJECT_ROOT, 'Manifest:', MANIFEST, 'Dataset:', DATASET_ROOT)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from cnn.dataset import discover_pairs, create_or_load_split_manifest
train_pairs = discover_pairs(DATASET_ROOT, 'our485', verify=True)
test_pairs = discover_pairs(DATASET_ROOT, 'eval15', verify=True)
manifest = create_or_load_split_manifest(DATASET_ROOT, MANIFEST, validation_fraction=0.15, seed=42)
counts = {name: sum(p['split'] == name for p in manifest['pairs']) for name in ('train', 'validation')}
print('our485 pairs:', len(train_pairs), counts, 'final-test pairs:', len(test_pairs))

In [ ]:
from cnn.train import parse_args, train

def run_training(mode, output, resume=None):
    argv = ['--dataset-root', str(DATASET_ROOT), '--manifest', str(MANIFEST),
            '--input-mode', mode, '--output-dir', str(output), '--device', 'auto',
            '--epochs', '50', '--batch-size', '8', '--learning-rate', '0.001']
    if mode == 'ga_clahe_refinement':
        argv += ['--hybrid-cache', str(WORK_ROOT / 'cache/ga_clahe_our485')]
    if resume: argv += ['--resume', str(resume)]
    return train(parse_args(argv))

if RUN_BASELINE:
    run_training('baseline', WORK_ROOT / 'models/baseline', RESUME_BASELINE)
if RUN_REFINEMENT:
    run_training('ga_clahe_refinement', WORK_ROOT / 'models/refinement', RESUME_REFINEMENT)

## Validation preview
The training loop already records original-resolution validation PSNR/SSIM. After a run, use the next cell for qualitative validation samples. It uses manifest validation files, never `eval15`.

In [ ]:
import cv2, matplotlib.pyplot as plt
from cnn.inference import load_checkpoint, enhance_with_cnn
validation_names = [p['name'] for p in manifest['pairs'] if p['split'] == 'validation'][:3]
checkpoint = WORK_ROOT / 'models/baseline/best.pt'
if checkpoint.is_file():
    model, metadata, _ = load_checkpoint(checkpoint, 'auto', 'baseline')
    fig, axes = plt.subplots(len(validation_names), 3, figsize=(12, 4 * len(validation_names)))
    for row, name in enumerate(validation_names):
        low = cv2.cvtColor(cv2.imread(str(DATASET_ROOT / 'our485/low' / name)), cv2.COLOR_BGR2RGB)
        high = cv2.cvtColor(cv2.imread(str(DATASET_ROOT / 'our485/high' / name)), cv2.COLOR_BGR2RGB)
        for ax, image, title in zip(axes[row], (low, enhance_with_cnn(low, model), high), ('Low', 'CNN', 'Target')):
            ax.imshow(image); ax.set_title(f'{name}: {title}'); ax.axis('off')
    plt.tight_layout()
else: print('Train or resume the baseline model to show comparisons.')

In [ ]:
# Final test is deliberately separate and opt-in. It must not guide model selection.
if RUN_FINAL_TEST:
    from cnn.evaluate import parse_args as parse_eval_args, evaluate
    baseline = WORK_ROOT / 'models/baseline/best.pt'
    refinement = WORK_ROOT / 'models/refinement/best.pt'
    argv = ['--dataset-root', str(DATASET_ROOT), '--output-dir', str(WORK_ROOT / 'final-test')]
    if baseline.is_file(): argv += ['--baseline-checkpoint', str(baseline)]
    if refinement.is_file(): argv += ['--refinement-checkpoint', str(refinement)]
    per_image, aggregate = evaluate(parse_eval_args(argv))
    display(aggregate)

## Retrieve and use checkpoints locally
After training, use **Save Version** so `/kaggle/working/clahe-cnn` becomes notebook output. Download `models/baseline/best.pt`, `models/refinement/best.pt`, histories, run configs, and the shared split manifest. Copy the weights into the local repository's corresponding `models/` folders and select them separately in Streamlit. Keep the manifest with the project so future experiments use the identical split.